In [ ]:
LABEL_DIR = r"C:\Project\Smart Parking\LocalMachine\Yolo\labels"                      # .txt labels (camera-prefixed)
IMG_BASE  = r"C:\Project\Smart Parking\LocalMachine\Yolo\dataset\FULL_IMAGE_1000x750" # images
WORK_DIR  = r"C:\Project\Smart Parking\LocalMachine\Yolo\yolo_splits"                 # temp split folder
YAML_PATH = "parking.yaml"

In [1]:
# ============================================================
# 🚗 YOLOv8 K-Fold Cross-Validation (Smart Parking)
#     ▶ Resumable • Cached • SafeGlobals (PyTorch 2.6+)
# ============================================================

import os, re, shutil, builtins, torch, pandas as pd
from tqdm import tqdm
from sklearn.model_selection import KFold

# ============================================================
# 🛠️ PyTorch 2.6+ Safe Load Setup (must run BEFORE YOLO import)
# ============================================================

# --- Core Torch modules ---
from torch.nn import (
    Conv2d, BatchNorm2d, ReLU, LeakyReLU, SiLU, Linear,
    Sequential, ModuleList, Upsample, MaxPool2d, AdaptiveAvgPool2d
)

# --- Explicit imports so ultralytics.nn.* exists ---
import ultralytics
import ultralytics.nn
import ultralytics.nn.tasks
import ultralytics.nn.modules
import ultralytics.nn.modules.block
import ultralytics.utils

# --- Register safe globals ---
torch.serialization.add_safe_globals([
    # Core Torch
    Conv2d, BatchNorm2d, ReLU, LeakyReLU, SiLU, Linear,
    Sequential, ModuleList, Upsample, MaxPool2d, AdaptiveAvgPool2d,

    # YOLO Core
    ultralytics.nn.tasks.DetectionModel,
    ultralytics.nn.modules.Conv,
    ultralytics.nn.modules.C2f,
    ultralytics.nn.modules.SPPF,
    ultralytics.nn.modules.Bottleneck,
    ultralytics.nn.modules.BottleneckCSP,
    ultralytics.nn.modules.Detect,

    # Modern YOLO Blocks
    ultralytics.nn.modules.block.DFL,
    ultralytics.nn.modules.block.C3k,
    ultralytics.nn.modules.block.C3k2,
    ultralytics.nn.modules.block.C2PSA,
    ultralytics.nn.modules.block.PSABlock,

    # Utility Namespaces
    ultralytics.utils.SimpleNamespace,
    ultralytics.utils.IterableSimpleNamespace,
])
print("✅ Safe globals registered")

# --- Force torch.load(weights_only=False) ---
if not getattr(builtins, "_torchload_force_weights_only_false", False):
    _orig_load = torch.load
    def _safe_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return _orig_load(*args, **kwargs)
    torch.load = _safe_load
    builtins._torchload_force_weights_only_false = True
print("✅ Torch hotfix applied (weights_only=False)")

# ============================================================
# 📦 Import YOLO after safe globals
# ============================================================
from ultralytics import YOLO

# ============================================================
# ⚙️ CONFIG
# ============================================================
LABEL_DIR = r"C:\Project\Smart Parking\LocalMachine\Yolo\labels"
IMG_BASE  = r"C:\Project\Smart Parking\LocalMachine\Yolo\dataset\FULL_IMAGE_1000x750"
WORK_DIR  = r"C:\Project\Smart Parking\LocalMachine\Yolo\yolo_splits"
YAML_PATH = "parking.yaml"

K_FOLDS   = 5
EPOCHS    = 50
BATCH     = 16
IMG_SIZE  = 640
RESULTS_CSV = "cv_results.csv"

# ============================================================
# 🧾 Collect labeled images
# ============================================================
images_list = []
for lbl in [f for f in os.listdir(LABEL_DIR) if f.endswith(".txt")]:
    name = lbl.replace(".txt", ".jpg")
    if name.startswith("camera"):
        name = "_".join(name.split("_")[1:])
    for root, _, files in os.walk(IMG_BASE):
        if name in files:
            images_list.append(os.path.join(root, name))
            break

print(f"🧾 Found {len(images_list)} labeled images for cross-validation")
if not images_list:
    raise RuntimeError("❌ No labeled images found — check paths!")

# ============================================================
# 🧩 Create YAML
# ============================================================
yaml_content = f"""
path: {WORK_DIR}
train: images/train
val: images/val

names:
  0: occupied
  1: free
"""
with open(YAML_PATH, "w") as f:
    f.write(yaml_content)

# ============================================================
# 🧩 Resume logic — detect completed folds
# ============================================================
completed_folds = set()
if os.path.exists(RESULTS_CSV):
    df_prev = pd.read_csv(RESULTS_CSV)
    completed_folds = set(df_prev["fold"].tolist())
    print(f"🔁 Detected completed folds: {sorted(completed_folds)}")
else:
    df_prev = pd.DataFrame(columns=["fold", "mAP50", "precision", "recall"])

# ============================================================
# 🔁 K-FOLD LOOP (resumable)
# ============================================================
kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(images_list), start=1):
    if fold in completed_folds:
        print(f"⏭️ Skipping Fold {fold} (already done)")
        continue

    print(f"\n🚀 Starting Fold {fold}/{K_FOLDS}")

    # Clean working dir for this fold
    shutil.rmtree(WORK_DIR, ignore_errors=True)
    for split in ["train", "val"]:
        os.makedirs(f"{WORK_DIR}/images/{split}", exist_ok=True)
        os.makedirs(f"{WORK_DIR}/labels/{split}", exist_ok=True)

    # --------------------------------------------------------
    # 📁 Copy Split
    # --------------------------------------------------------
    def copy_split(indices, split):
        skipped, matched = 0, 0
        for i in tqdm(indices, desc=f"Copying {split} data"):
            img = images_list[i]
            base = os.path.basename(img).replace(".jpg", "")
            cam_folder = os.path.basename(os.path.dirname(img))
            pattern = f"{cam_folder}_{base}.txt"
            lbl = os.path.join(LABEL_DIR, pattern)

            if not os.path.exists(lbl):
                skipped += 1
                continue

            try:
                shutil.copy(img, f"{WORK_DIR}/images/{split}/")
                shutil.copy(lbl, f"{WORK_DIR}/labels/{split}/")
                matched += 1
            except Exception:
                skipped += 1
        print(f"✅ Copied {matched} for {split}. ⚠️ Skipped {skipped}.")

    copy_split(train_idx, "train")
    copy_split(val_idx, "val")

    # --------------------------------------------------------
    # 🧠 Rename (camera-safe)
    # --------------------------------------------------------
    for split in ["train", "val"]:
        img_dir = os.path.join(WORK_DIR, "images", split)
        lbl_dir = os.path.join(WORK_DIR, "labels", split)
        for img_file in os.listdir(img_dir):
            base = os.path.splitext(img_file)[0]
            cam_folder = None
            for cam in [f"camera{i}" for i in range(1, 10)]:
                if os.path.exists(os.path.join(LABEL_DIR, f"{cam}_{base}.txt")):
                    cam_folder = cam
                    break
            if cam_folder:
                new_img = f"{cam_folder}_{img_file}"
                new_lbl = f"{cam_folder}_{base}.txt"
                try:
                    os.rename(os.path.join(img_dir, img_file), os.path.join(img_dir, new_img))
                    if os.path.exists(os.path.join(lbl_dir, f"{base}.txt")):
                        os.rename(os.path.join(lbl_dir, f"{base}.txt"), os.path.join(lbl_dir, new_lbl))
                except Exception as e:
                    print(f"⚠️ Rename skipped for {img_file}: {e}")

    # --------------------------------------------------------
    # 🧠 Train YOLO
    # --------------------------------------------------------
    model = YOLO("yolov8n.pt")
    hist = model.train(
        data=YAML_PATH,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        project="runs",
        name=f"fold_{fold}",
        save=True,                # ✅ save weights
        exist_ok=True,
        device=0 if torch.cuda.is_available() else "cpu",
        cache=True,               # ✅ caching accelerates subsequent folds
        amp=False,                # ✅ disable AMP to skip auto reload
        workers=4
    )

    # --------------------------------------------------------
    # 📈 Record + Save Results
    # --------------------------------------------------------
    fold_result = {
        "fold": fold,
        "mAP50": hist.results_dict.get("metrics/mAP50(B)", None),
        "precision": hist.results_dict.get("metrics/precision(B)", None),
        "recall": hist.results_dict.get("metrics/recall(B)", None)
    }

    # Append safely to CSV
    mode = "a" if os.path.exists(RESULTS_CSV) else "w"
    header = not os.path.exists(RESULTS_CSV)
    pd.DataFrame([fold_result]).to_csv(RESULTS_CSV, mode=mode, index=False, header=header)
    print(f"💾 Saved fold {fold} results to {RESULTS_CSV}")

print("\n✅ Cross-validation complete (resumable + cached + safe)")

# ============================================================
# 📊 Final Combined Results
# ============================================================
if os.path.exists(RESULTS_CSV):
    df = pd.read_csv(RESULTS_CSV)
    print("\n📊 All Fold Results:")
    print(df)
    print("\n🏁 Mean Performance:")
    print(df.mean(numeric_only=True))


✅ Safe globals registered
✅ Torch hotfix applied (weights_only=False)
🧾 Found 4081 labeled images for cross-validation
🔁 Detected completed folds: [1, 2, 3, 4]
⏭️ Skipping Fold 1 (already done)
⏭️ Skipping Fold 2 (already done)
⏭️ Skipping Fold 3 (already done)
⏭️ Skipping Fold 4 (already done)

🚀 Starting Fold 5/5


Copying train data: 100%|█████████████████████████████████████████████████████████| 3265/3265 [00:07<00:00, 433.71it/s]


✅ Copied 3259 for train. ⚠️ Skipped 6.


Copying val data: 100%|█████████████████████████████████████████████████████████████| 816/816 [00:01<00:00, 494.32it/s]


✅ Copied 814 for val. ⚠️ Skipped 2.
Ultralytics 8.3.223  Python-3.12.6 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=parking.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=fold_5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=